# 06 · Training decoders

The networks are frozen optical-flow models; the task readouts are small MLPs
trained on their activity — one hit/miss classifier per network, plus position
and velocity regressors.

Classifier: 512 tanh units, Adam at 1e-3, batch 1024, L2 0.1, up to 50 epochs,
early stopping after 10 without improvement. Regressors: 128-16 ReLU units,
MSE, up to 200 epochs, early stopping after 50.

## 1. Generate the activity

`save_simulations` renders the ~6,000 trials in
`data/configs/train_configuration.pt` and stores what the network feeds its
decoder. It takes `(net, network_id)`, so it works for every architecture — run
it once per network.

In [ ]:
import torch
from flynet_robustness import config
from flynet_robustness.models import load_network
from flynet_robustness.dataset import save_simulations, prepare_dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
network_id = 'FlyNet'      # any id

net, _ = load_network(network_id, device=device, decoder=False)
# hours on a GPU; run once, then reload from disk
# save_simulations(net, network_id, config_path='data/configs/train_configuration.pt',
#                  out_dir=f'simulations/{network_id}', device=device)
activity = torch.load(f'simulations/{network_id}/activity.pt',
                      weights_only=False)['network_activity_all']
print(len(activity), 'trials, activity', tuple(activity[0].shape))

## 2. Hit/miss classifier

Trials, not timepoints, are split into train and test — timepoints from one
trial are far too similar to sit on both sides. Accuracy is reported in 20 bins
of normalised time, since the decision is easy late and hard early.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from flynet_robustness.training import train_hit_decoder

cfg = torch.load('data/configs/train_configuration.pt', weights_only=False)
acc, test_acc = train_hit_decoder(activity, cfg['true_labels'],
                                  cfg['time_duration_conditions'],
                                  network_id=network_id, decoder='mlp', scale=True)
print('test accuracy', round(test_acc, 3))
plt.plot(np.linspace(0, 1, len(acc)), acc, color=config.color(network_id))
plt.axhline(0.5, ls='--', c='gray'); plt.xlabel('normalised time'); plt.ylabel('test accuracy');

## 3. Position and velocity

`prepare_dataset` stacks the trials and trims each one's positions to the
number of samples that network produced, so pass the network id.

In [ ]:
from flynet_robustness.training import train_position_decoder

positions = torch.load(f'simulations/{network_id}/positions.pt',
                       weights_only=False)['positions']
X, Y = prepare_dataset(activity, positions, network_id=network_id)
print('X', tuple(X.shape), 'Y', tuple(Y.shape))

velocity = [torch.gradient(p, axis=1)[0] for p in positions]
Xv, Yv = prepare_dataset(activity, velocity, network_id=network_id)

# train_position_decoder(X, Y, network_id=network_id, velocity=False, device=device)
# train_position_decoder(Xv, Yv, network_id=network_id, velocity=True, device=device)